In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Model Persistence notebook (Hardening Step 2).
Single consolidated code cell (platform convention). Idempotent - safe to re-run.

Mirrors BP1's and BP2's model-persistence notebooks in structure, adapted for BP3's real
structured-feature pipeline (one-hot + company-frequency preprocessing outside a single sklearn
Pipeline - see src/features/bp3_escalation_features.py and src/models/model_persistence.py's
module docstring for why BP3's persisted bundle is a plain dict of {preprocessor,
company_freq_map, classifier, ...} rather than one self-contained sklearn Pipeline object, unlike
BP1's). Run only after BP3 Gate 5 (Decision Layer & Reporting) is real-run confirmed: it refits
the already-confirmed xgboost champion on the identical full-train split Gate 5 already validated,
persists it to models/bp3_complaint_escalation_prediction/, and proves the persisted artifact is a
faithful copy by reloading it and reproducing Gate 5's exact recorded PR-AUC/recall.

BP3 differs from BP2's model-persistence notebook in two real, load-bearing ways: (1) BP3's target
(intervention_required) is already a binary 0/1 int column in the real Gold layer (cast via Polars
.cast(pl.Int8) at Gate 3, never passed through a LabelEncoder anywhere in the real pipeline) - so
this notebook fits no LabelEncoder and the persisted bundle has no label_encoder key, matching
model_persistence.py's REQUIRED_KEYS["bp3"]. (2) BP3's champion-selection/fidelity metric is
PR-AUC (average_precision_score) and recall at the 0.5 decision threshold, never accuracy - an
explicit Master Plan BP3 methodology rule stated in the real Gate 3 notebook's own comments and
reproduced verbatim here as the fidelity check.

Reuses src/features/bp3_escalation_features.py (Gate 6's own extraction) for candidate/feature
definitions AND for build_shared_preprocessing() (BP3's Gate 6 module exports this shared
preprocessing builder, unlike BP2's equivalent module - so this notebook calls it directly rather
than re-inlining the OHE+frequency construction a further time, HYPER: single source of truth) and
src/models/model_persistence.py (extended this step with a "bp3" bundle contract and predict_bp3())
for save/load/predict mechanics.
"""

import os, sys, json, time, platform, warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.metadata  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import sklearn  # noqa: E402
import yaml  # noqa: E402
from sklearn.metrics import average_precision_score, recall_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

print = functools.partial(builtins.print, flush=True)

from features.bp3_escalation_features import (  # noqa: E402
    BARRED_COLUMNS,
    COMPANY_COL,
    FEATURE_COLS_CATEGORICAL,
    NEEDS_DENSE,
    build_shared_preprocessing,
    make_candidates,
)
from models.model_persistence import load_model_bundle, predict_bp3, save_model_bundle  # noqa: E402

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models" / "bp3_complaint_escalation_prediction"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"
BP3_CONFIG_PATH = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"

for p in (GOLD_PATH, BP3_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(f"Required input not found: {p}. Confirm BP3 Gates 1-5 all completed for real.")

# ============================================================
# SECTION 4: Load Gate 3/4/5's real results - champion read LIVE, never hardcoded. Requires
# Gate 5 (not just Gate 4) to already be real-run confirmed, since this step persists the exact
# fitted state Gate 5 already validated end to end (same full-train refit, same PR-AUC/recall).
# ============================================================
with open(BP3_CONFIG_PATH, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

TARGET_COL = bp3_config["target_definition"]["primary_target"]
RANDOM_STATE = bp3_config["random_state"]

gate3_block = bp3_config.get("gate3_model_benchmark")
assert gate3_block is not None, "[CHECK FAILED] gate3_model_benchmark missing - run BP3 Gate 3 first."
gate5_block = bp3_config.get("gate5_decision_layer")
assert gate5_block is not None, (
    "[CHECK FAILED] gate5_decision_layer missing from configs/bp3_complaint_escalation_prediction.yaml - "
    "run BP3 Gate 5 (Decision Layer & Reporting) for real before this model-persistence step."
)

gate4_json_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
assert gate4_json_path.exists(), f"[CHECK FAILED] {gate4_json_path} not found - run BP3 Gate 4 first."
with open(gate4_json_path, "r", encoding="utf-8") as f:
    gate4_results = json.load(f)
CHAMPION_NAME = gate4_results["champion_model"]
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: Gate 4 recorded '{CHAMPION_NAME}' but Gate 3's config block says "
    f"'{gate3_block['champion_model']}' - these must agree; re-run Gate 3/4."
)

gate5_json_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
assert gate5_json_path.exists(), f"[CHECK FAILED] {gate5_json_path} not found - run BP3 Gate 5 first."
with open(gate5_json_path, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)
assert gate5_summary["champion_model"] == CHAMPION_NAME, (
    f"[CHECK FAILED] Gate 5 recorded champion '{gate5_summary['champion_model']}' does not match "
    f"Gate 4's '{CHAMPION_NAME}' - re-run Gate 5."
)
gate5_recorded_pr_auc = float(gate5_summary["held_out_test_pr_auc_recomputed"])
gate3_recorded_pr_auc = float(gate3_block["held_out_test_pr_auc"])
gate3_recorded_recall = float(gate3_block["held_out_test_recall"])
print(f"[OK] Champion (live, re-verified against Gate 3 + Gate 4 + Gate 5): {CHAMPION_NAME}")

# ============================================================
# SECTION 5: Rebuild the real Gold-layer feature frame EXACTLY as Gate 3 did (HYPER reuse of
# src/features/bp3_escalation_features.py's own column constants). Trainable rows = TARGET_COL not
# null (BP3's exclusion rule is null-based, not a fixed class list like BP2's ordinal severity).
# ============================================================
gold_lazy = pl.scan_parquet(GOLD_PATH)
select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL]
df_pl = gold_lazy.select(select_cols).filter(pl.col(TARGET_COL).is_not_null()).collect()
for barred in BARRED_COLUMNS:
    assert barred not in df_pl.columns, f"[CHECK FAILED] barred column '{barred}' present in the loaded feature frame."
print(f"[OK] Reloaded real Gold layer, trainable rows: {df_pl.height:,}.")

# Gate 2's own null-sentinel fill already covers FEATURE_COLS_CATEGORICAL on the Gold layer (see
# bp3_escalation_features.fill_categorical_nulls_expr()) - only COMPANY_COL gets a defensive
# fill_null here, matching Gate 3's own real reload code exactly (never adding a fill Gate 3 itself
# does not perform, which would silently diverge from the already-validated fitted state).
feature_data = {col: df_pl[col].cast(pl.Utf8).to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Int8).to_list()
X_full = pd.DataFrame(feature_data)
y_full = pd.Series(target_data, name=TARGET_COL)

# ============================================================
# SECTION 6: Identical stratified train/test split as Gates 3/4/5. No LabelEncoder anywhere - the
# real target is already binary 0/1 (see module docstring), unlike BP1's/BP2's multi-class targets.
# ============================================================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE
)
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
print(f"[OK] Reproduced Gate 3/4/5's train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,}.")

n_train_positive = int((y_train == 1).sum())
n_train_negative = int((y_train == 0).sum())
scale_pos_weight = n_train_negative / n_train_positive
print(
    f"[OK] Live train-split class ratio: {n_train_negative:,} negative / {n_train_positive:,} "
    f"positive -> scale_pos_weight={scale_pos_weight:.2f} (computed live, never a guessed constant)."
)

# ============================================================
# SECTION 7: Rebuild the shared preprocessing EXACTLY as Gates 3/4/5 (fit on TRAIN only) via
# src/features/bp3_escalation_features.py's own build_shared_preprocessing() - BP3's Gate 6 module
# exports this shared builder (unlike BP2's equivalent module, which this notebook's BP2 sibling
# re-inlines), so it is reused directly here rather than triplicated a further time (HYPER).
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
preprocessor, X_train_shared, X_test_shared, company_freq_map, feature_names = build_shared_preprocessing(
    X_train_raw, X_test_raw
)
print(f"[OK] Rebuilt shared OHE+frequency preprocessing (fit on train only), shape={X_train_shared.shape}.")

candidates = make_candidates(random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight)
champion_model = candidates[CHAMPION_NAME]
needs_dense = CHAMPION_NAME in NEEDS_DENSE

X_train_final, X_test_final = X_train_shared, X_test_shared
if needs_dense:
    X_train_final = np.asarray(X_train_final.todense(), dtype=np.float32)
    X_test_final = np.asarray(X_test_final.todense(), dtype=np.float32)

print(f"\n[PERSIST] Refitting champion ({CHAMPION_NAME}) on the full train split...")
t0 = time.perf_counter()
champion_model.fit(X_train_final, y_train)
fit_seconds = time.perf_counter() - t0
print(f"[PERSIST] Fit done in {fit_seconds:.1f}s")

# PR-AUC (average_precision_score) + recall at the 0.5 default threshold - the real BP3
# champion-selection/fidelity metric (never accuracy, an explicit Master Plan BP3 methodology
# rule the real Gate 3 notebook's own comments state, reproduced verbatim here).
y_proba_test = champion_model.predict_proba(X_test_final)[:, 1]
y_pred_test = (y_proba_test >= 0.5).astype(int)
fresh_pr_auc = float(average_precision_score(y_test, y_proba_test))
fresh_recall = float(recall_score(y_test, y_pred_test))
diff_vs_gate3_pr_auc = abs(fresh_pr_auc - gate3_recorded_pr_auc)
diff_vs_gate5_pr_auc = abs(fresh_pr_auc - gate5_recorded_pr_auc)
diff_vs_gate3_recall = abs(fresh_recall - gate3_recorded_recall)
print(f"[CHECK] Fresh refit test PR-AUC: {fresh_pr_auc:.6f} "
      f"(Gate 3 recorded: {gate3_recorded_pr_auc:.6f}, diff={diff_vs_gate3_pr_auc:.6f}; "
      f"Gate 5 recomputed: {gate5_recorded_pr_auc:.6f}, diff={diff_vs_gate5_pr_auc:.6f})")
print(f"[CHECK] Fresh refit test recall @0.5: {fresh_recall:.6f} "
      f"(Gate 3 recorded: {gate3_recorded_recall:.6f}, diff={diff_vs_gate3_recall:.6f})")

# ============================================================
# SECTION 8: Assemble and persist the model bundle (src/models/model_persistence.py). BP3's
# bundle is a plain dict, not one sklearn Pipeline, and carries NO label_encoder key - see
# model_persistence.py's module docstring for why (BP3's target is already binary 0/1, never
# passed through a LabelEncoder anywhere in the real pipeline).
# ============================================================
generated_at = datetime.now(timezone.utc).isoformat()
bundle = {
    "bp_id": "bp3",
    "champion_name": CHAMPION_NAME,
    "preprocessor": preprocessor,
    "company_freq_map": company_freq_map,
    "feature_cols_categorical": FEATURE_COLS_CATEGORICAL,
    "company_col": COMPANY_COL,
    "classifier": champion_model,
    "class_names": ["0", "1"],
    "needs_dense": needs_dense,
    "metadata": {
        "gate3_recorded_test_pr_auc": gate3_recorded_pr_auc,
        "gate3_recorded_test_recall": gate3_recorded_recall,
        "gate5_recomputed_test_pr_auc": gate5_recorded_pr_auc,
        "fresh_refit_test_pr_auc": fresh_pr_auc,
        "fresh_refit_test_recall": fresh_recall,
        "n_train_rows": int(len(X_train_raw)),
        "n_test_rows": int(len(X_test_raw)),
        "scale_pos_weight": round(scale_pos_weight, 4),
        "random_state": RANDOM_STATE,
        "fit_seconds": round(fit_seconds, 2),
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__,
        "joblib_version": importlib.metadata.version("joblib"),
        "generated_at_utc": generated_at,
    },
}

out_path = MODELS_DIR / "bp3_champion_bundle.joblib"
save_stats = save_model_bundle(bundle, out_path)
print(f"\n[SAVED] {out_path.relative_to(PROJECT_ROOT)} "
      f"({save_stats['size_bytes']:,} bytes, sha256={save_stats['sha256'][:16]}...)")

# ============================================================
# SECTION 9: Reload the persisted bundle and prove it is a faithful, usable copy - not merely
# that the file exists. Also verifies the unseen-company inference fallback (frequency=0, never
# fabricated) on a real held-out row with its Company value swapped to a name never seen in
# training - the exact edge case a production inference service will hit routinely.
# ============================================================
reloaded_bundle = load_model_bundle(out_path)
reload_result = predict_bp3(reloaded_bundle, X_test_raw)
reloaded_proba = np.array(reload_result["probability_positive_class"])
reloaded_pred = np.array(reload_result["predicted_label"])
reloaded_pr_auc = float(average_precision_score(y_test, reloaded_proba))
reloaded_recall = float(recall_score(y_test, reloaded_pred))
reload_pr_auc_diff = abs(reloaded_pr_auc - fresh_pr_auc)
reload_recall_diff = abs(reloaded_recall - fresh_recall)
print(f"[CHECK] Reloaded-bundle PR-AUC via predict_bp3(): {reloaded_pr_auc:.6f} "
      f"(fresh in-memory: {fresh_pr_auc:.6f}, diff={reload_pr_auc_diff:.10f})")
print(f"[CHECK] Reloaded-bundle recall via predict_bp3(): {reloaded_recall:.6f} "
      f"(fresh in-memory: {fresh_recall:.6f}, diff={reload_recall_diff:.10f})")

unseen_row = X_test_raw.iloc[[0]].copy()
unseen_row[COMPANY_COL] = "A Totally Fictional Company Never In Training Data LLC"
unseen_result = predict_bp3(reloaded_bundle, unseen_row)
unseen_proba_sum = sum(unseen_result["probabilities"][0])
print(f"[CHECK] Unseen-company inference (frequency=0 fallback) probability row sums to {unseen_proba_sum:.6f}.")

# ============================================================
# SECTION 10: Write a human-readable metadata sidecar (the .joblib itself is gitignored per this
# project's own .gitignore - models/**/*.joblib - so this JSON is the artifact's committed,
# version-controlled record of what was persisted and how it was verified).
# ============================================================
metadata_record = {
    "bp_id": "bp3",
    "champion_model": CHAMPION_NAME,
    "joblib_relative_path": str(out_path.relative_to(PROJECT_ROOT)),
    "joblib_size_bytes": save_stats["size_bytes"],
    "joblib_sha256": save_stats["sha256"],
    "needs_dense": needs_dense,
    "fresh_refit_test_pr_auc": round(fresh_pr_auc, 6),
    "fresh_refit_test_recall": round(fresh_recall, 6),
    "gate3_recorded_test_pr_auc": round(gate3_recorded_pr_auc, 6),
    "gate3_recorded_test_recall": round(gate3_recorded_recall, 6),
    "gate5_recomputed_test_pr_auc": round(gate5_recorded_pr_auc, 6),
    "pr_auc_diff_vs_gate3": round(diff_vs_gate3_pr_auc, 6),
    "pr_auc_diff_vs_gate5": round(diff_vs_gate5_pr_auc, 6),
    "recall_diff_vs_gate3": round(diff_vs_gate3_recall, 6),
    "reload_verified_pr_auc": round(reloaded_pr_auc, 6),
    "reload_verified_recall": round(reloaded_recall, 6),
    "reload_pr_auc_diff": round(reload_pr_auc_diff, 10),
    "reload_recall_diff": round(reload_recall_diff, 10),
    "unseen_company_inference_probability_sum": round(unseen_proba_sum, 6),
    "bundle_keys": sorted(bundle.keys()),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "joblib_version": importlib.metadata.version("joblib"),
    "generated_at_utc": generated_at,
}
metadata_path = MODELS_DIR / "bp3_model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_record, f, indent=2)
print(f"[SAVED] {metadata_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 11: Update the established model_inventory_entry.json (idempotent, same pattern every
# other gate already uses) and the BP config YAML's marker-delimited block.
# ============================================================
inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
with open(inventory_path, "r", encoding="utf-8") as f:
    model_inventory_entry = json.load(f)
model_inventory_entry["status"] = "Model persistence (Hardening Step 2) complete"
model_inventory_entry["model_persistence_joblib_path"] = str(out_path.relative_to(PROJECT_ROOT))
model_inventory_entry["model_persistence_sha256"] = save_stats["sha256"]
model_inventory_entry["model_persistence_reload_verified"] = (
    reload_pr_auc_diff < 1e-9 and reload_recall_diff < 1e-9
)
model_inventory_entry["model_persistence_generated_at_utc"] = generated_at
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (model_persistence fields added)")

from utils.bp1_config_sync import write_gate_block  # noqa: E402 - generic helper, reused across BPs

persistence_marker = "# --- Model Persistence (Hardening Step 2) results (appended, idempotent overwrite) ---"
persistence_block_lines = [
    "model_persistence:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f'  joblib_relative_path: "{out_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'  joblib_sha256: "{save_stats["sha256"]}"',
    f"  fresh_refit_test_pr_auc: {round(fresh_pr_auc, 6)}",
    f"  fresh_refit_test_recall: {round(fresh_recall, 6)}",
    f"  reload_verified_pr_auc: {round(reloaded_pr_auc, 6)}",
    f"  reload_verified_recall: {round(reloaded_recall, 6)}",
    f"  reload_pr_auc_diff: {round(reload_pr_auc_diff, 10)}",
    f"  reload_recall_diff: {round(reload_recall_diff, 10)}",
    f'  generated_at_utc: "{generated_at}"',
]
write_gate_block(BP3_CONFIG_PATH, persistence_marker, persistence_block_lines)
print(f"[SAVED] {BP3_CONFIG_PATH.relative_to(PROJECT_ROOT)} (model_persistence block)")

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_matches_gate3_gate4_gate5_recorded": CHAMPION_NAME == gate3_block["champion_model"] == gate5_summary["champion_model"],
    "fresh_refit_pr_auc_matches_gate3_recorded": diff_vs_gate3_pr_auc < 1e-3,
    "fresh_refit_pr_auc_matches_gate5_recomputed": diff_vs_gate5_pr_auc < 1e-2,
    "fresh_refit_recall_matches_gate3_recorded": diff_vs_gate3_recall < 1e-3,
    "joblib_file_written": out_path.exists(),
    "joblib_file_nonempty": save_stats["size_bytes"] > 0,
    "reload_round_trip_pr_auc_matches_fresh_fit": reload_pr_auc_diff < 1e-9,
    "reload_round_trip_recall_matches_fresh_fit": reload_recall_diff < 1e-9,
    "reload_predictions_cover_full_test_set": len(reloaded_pred) == len(X_test_raw),
    "reload_probabilities_sum_to_one": all(
        abs(sum(row) - 1.0) < 1e-6 for row in reload_result["probabilities"][:50]
    ),
    "unseen_company_inference_valid_probability_row": abs(unseen_proba_sum - 1.0) < 1e-6,
    "bundle_has_all_required_keys": {
        "bp_id", "champion_name", "preprocessor", "company_freq_map", "feature_cols_categorical",
        "company_col", "classifier", "class_names", "needs_dense", "metadata",
    }.issubset(bundle.keys()),
    "bundle_has_no_label_encoder_key": "label_encoder" not in bundle,
    "metadata_sidecar_written": metadata_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp3_config_yaml_updated": BP3_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP3 model persistence complete. Champion ({CHAMPION_NAME}) persisted to "
      f"{out_path.relative_to(PROJECT_ROOT)} ({save_stats['size_bytes']:,} bytes). Reload-verified "
      f"PR-AUC {round(reloaded_pr_auc, 4)} and recall {round(reloaded_recall, 4)} exactly match the "
      f"fresh refit and stay consistent with Gate 5's already-confirmed recomputed PR-AUC "
      f"{round(gate5_recorded_pr_auc, 4)}. Ready for Hardening Step 3 (FastAPI inference service).")
